In [6]:
import os
import torch
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
from tqdm import tqdm
from pathlib import Path

# ================= 配置区域 =================

# 路径配置（保持和你之前的结构一致）
DATASET_ROOT = "./dataset_tri_RAD_Tensor"
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test") # 或者是 sim_val
RESULTS_DIR = "./training_results_tensor"
ONNX_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model_int8.onnx")
CM_SAVE_PATH = os.path.join(RESULTS_DIR, "onnx_confusion_matrix.png")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK",
    "16QAM", "32QAM", "64QAM",
    "2ASK", "16APSK", "32APSK", "FSK2"
]

# ================= 辅助函数 =================

def to_numpy(tensor):
    """将 PyTorch Tensor 转换为 Numpy array"""
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    """绘制混淆矩阵"""
    # 动态筛选存在的类别（防止某些类别没有数据导致报错）
    labels = sorted(list(set(y_true) | set(y_pred)))
    target_names = [CLASS_NAMES[i] for i in labels]
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    print(f"Confusion matrix saved to {save_path}")
    plt.close()

# ================= 主推理逻辑 =================

def main():
    # 1. 检查模型是否存在
    if not os.path.exists(ONNX_MODEL_PATH):
        print(f"Error: ONNX model not found at {ONNX_MODEL_PATH}")
        print("Please run the export step first.")
        return

    # 2. 初始化 ONNX Runtime 会话
    print(f"Loading ONNX model from {ONNX_MODEL_PATH}...")
    # providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] 如果装了GPU版会自动用GPU
    ort_session = ort.InferenceSession(ONNX_MODEL_PATH, providers=['CPUExecutionProvider'])
    
    # 获取输入输出节点的名称
    input_name = ort_session.get_inputs()[0].name
    output_name = ort_session.get_outputs()[0].name
    
    input_shape = ort_session.get_inputs()[0].shape
    print(f"Model Input Name: {input_name}, Shape: {input_shape}")

    # 3. 加载测试数据路径
    if not os.path.exists(REAL_TEST_DIR):
        print(f"Test data directory not found: {REAL_TEST_DIR}")
        return

    test_files = []
    print(f"Scanning files in {REAL_TEST_DIR}...")
    for label_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = Path(REAL_TEST_DIR) / cls_name
        if cls_dir.exists():
            files = list(cls_dir.glob("*.pt"))
            # 将 (文件路径, 真实标签索引) 加入列表
            test_files.extend([(p, label_idx) for p in files])
    
    if len(test_files) == 0:
        print("No .pt files found for testing.")
        return

    print(f"Found {len(test_files)} samples. Starting inference...")

    # 4. 执行推理
    y_true = []
    y_pred = []

    for file_path, label in tqdm(test_files, unit="sample"):
        try:
            # A. 加载数据
            # 这里加载的是 (3, 64, 64) 的 tensor
            tensor_data = torch.load(file_path)
            
            # B. 预处理
            # ONNX 需要 numpy 格式
            # 模型输入需要 batch 维度 -> (1, 3, 64, 64)
            numpy_input = to_numpy(tensor_data).astype(np.float32)
            numpy_input = np.expand_dims(numpy_input, axis=0) 
            
            # C. 推理 (Run)
            # ort_inputs = {input_name: input_data}
            outputs = ort_session.run([output_name], {input_name: numpy_input})
            
            # D. 获取结果
            # outputs[0] 是 logits, shape (1, 12)
            scores = outputs[0]
            predicted_idx = np.argmax(scores, axis=1)[0]
            
            y_true.append(label)
            y_pred.append(predicted_idx)
            
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # 5. 结果分析
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    acc = accuracy_score(y_true, y_pred)
    print(f"\n================ Results ================")
    print(f"Total Samples: {len(y_true)}")
    print(f"Accuracy:      {acc:.2%}")
    print(f"=========================================")

    # 6. 绘制混淆矩阵
    plot_confusion_matrix(
        y_true, 
        y_pred, 
        f"ONNX Inference Result (Acc: {acc:.2%})", 
        CM_SAVE_PATH
    )

if __name__ == "__main__":
    main()

Loading ONNX model from ./training_results_tensor\best_model_int8.onnx...
Model Input Name: input, Shape: ['batch_size', 3, 64, 64]
Scanning files in ./dataset_tri_RAD_Tensor\real_test...
Found 43938 samples. Starting inference...


100%|██████████| 43938/43938 [00:25<00:00, 1732.16sample/s]



================ Results ================
Total Samples: 43938
Accuracy:      100.00%
Confusion matrix saved to ./training_results_tensor\onnx_confusion_matrix.png
